In [ ]:
import numpy as np
import math

In [ ]:
param_ranges = {
    'b': (200.0, 500.0),
    'd': (300.0, 1500.0),
    'Ast': (200.0, 10000.0)
}

In [ ]:
# === Helper Functions ===
def get_xul(d, fy):
    Es = 2e5
    return round(0.0035 * d / (0.0055 + 0.87 * fy / Es), 2)

def calculate_moment_capacity(b, d, Ast, fck, fy):
    xu = (0.87 * fy * Ast) / (0.36 * fck * b)
    MOR = 0.87 * fy * Ast * (d - 0.416 * xu) / 1e6
    xul = get_xul(d, fy)
    Mul = 0.36 * fck * b * xul * (d - 0.416 * xul) / 1e6
    return round(MOR, 3), round(Mul, 3)

def Ast_limits(b, d, fy):
    Ast_min = 0.85 * b * d / fy
    Ast_max = 0.04 * b * d
    return round(Ast_min, 2), round(Ast_max, 2)

def bd_ratio(b, d):
    return b / d

def get_cost(b, d, Ast):
    concrete_vol = (b * d) * 1e-6  # m³/m
    steel_wt = Ast * 1e-6 * 7850   # kg/m
    return round(concrete_vol * 15000 + steel_wt * 140,2)

def estimate_cost_bounds(param_ranges):
    from itertools import product
    extremes = [
        (param_ranges['b'][i], param_ranges['d'][j], param_ranges['Ast'][k])
        for i in [0, 1]
        for j in [0, 1]
        for k in [0, 1]
    ]
    costs = [get_cost(b, d, Ast) for b, d, Ast in extremes]
    return min(costs), max(costs)

def get_penalty(rel_violation, weight):
    return weight * rel_violation

def get_reward(b, d, Ast, fck, fy, Mu):
    W_AST = 75
    W_BD = 75
    W_SAFETY = 100
    W_OVER_REINFORCE = 100
    W_COST = 50

    Ast_min, Ast_max = Ast_limits(b, d, fy)
    bd_rat = bd_ratio(b, d)
    MOR, Mul = calculate_moment_capacity(b, d, Ast, fck, fy)

    penalty = 0.0
    if Ast < Ast_min:
        penalty += get_penalty((Ast_min - Ast) / Ast_min, W_AST)
    if Ast > Ast_max:
        penalty += get_penalty((Ast - Ast_max) / Ast_max, W_AST)
    if bd_rat < 0.3:
        penalty += get_penalty((0.3 - bd_rat) / 0.3, W_BD)
    if Mu >= MOR:
        penalty += get_penalty((Mu - MOR) / Mu, W_SAFETY)
    if MOR >= Mul:
        penalty += get_penalty((MOR - Mul) / max(Mul, 1e-6), W_OVER_REINFORCE)

    cost = get_cost(b, d, Ast)
    cost_min, cost_max = estimate_cost_bounds(param_ranges)
    # cost_score = (cost_max - cost) / (cost_max - cost_min)
    cost_score = np.log(cost_max / cost) / np.log(cost_max / cost_min)
    cost_score = max(0, min(1, cost_score))

    reward = W_COST * cost_score
    return reward - penalty

def is_valid_design(b, d, Ast, fck, fy, Mu, verbose=False):
    Ast_min, Ast_max = Ast_limits(b, d, fy)

    if not (Ast_min <= Ast <= Ast_max):
        if verbose:
            if Ast < Ast_min:
                print(
                    f"Design Invalid: Steel area Ast ({Ast}) is less than minimum required Ast_min ({Ast_min})."
                )
            else:
                print(
                    f"Design Invalid: Steel area Ast ({Ast}) exceeds maximum allowed Ast_max ({Ast_max})."
                )
        return False

    ratio = bd_ratio(b, d)
    if ratio < 0.3:
        if verbose:
            print(
                f"Design Invalid: Width-to-depth ratio b/d ({ratio:.3f}) is less than 0.3."
            )
        return False

    MOR, Mul = calculate_moment_capacity(b, d, Ast, fck, fy)

    if Mu > MOR:
        if verbose:
            print(
                f"Design Invalid: Applied moment Mu ({Mu}) exceeds moment capacity MOR ({MOR})."
            )
        return False

    if MOR > Mul:
        if verbose:
            print(
                f"Design Invalid: Section is over-reinforced (MOR {MOR} > Mul {Mul})."
            )
        return False

    return True



In [ ]:
def print_accuracy(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    print(f"R²: {r2:.4f}, RMSE: {rmse:.2f}")

def get_accuracy(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    return round(r2,4),  round(rmse,4)

def get_cost_accuracy(cost_true, cost_pred):
    cost_accuracy = 1 - abs(cost_pred - cost_true) / cost_true
    return cost_accuracy

def get_costs_accuracy(costs_true, costs_pred):
    cost_accuracy = np.mean([get_cost_accuracy(true, pred) for true, pred in zip(costs_true, costs_pred)])
    return np.round(cost_accuracy,4)

def evaluate_model(y_test, y_pred, target_names=['b', 'd', 'Ast']):
    for i, name in enumerate(target_names):
        r2 = r2_score(y_test[:, i], y_pred[:, i])
        rmse = root_mean_squared_error(y_test[:, i], y_pred[:, i])
        print(f"{name}: R² = {r2:.4f}, RMSE = {rmse:.2f}")

    costs_true = [get_cost(b,d,Ast) for b,d,Ast in zip(y_test[:,0], y_test[:,1], y_test[:,2])]
    costs_pred = [get_cost(b,d,Ast) for b,d,Ast in zip(y_pred[:,0], y_pred[:,1], y_pred[:,2])]

    print("Cost accuracy:", get_costs_accuracy(costs_true, costs_pred))


def plot_predictions(y_test, y_pred, target_names=['b', 'd', 'Ast'], model="LR"):
    for i, name in enumerate(target_names):
        # Accuracy metrics
        r2, rmse = get_accuracy(y_test[:, i], y_pred[:, i])
        textstr = f"R² = {r2:.4f}\nRMSE = {rmse:.2f}"

        plt.figure(figsize=(5, 4))
        plt.scatter(y_test[:, i], y_pred[:, i], alpha=0.6)
        plt.plot([y_test[:, i].min(), y_test[:, i].max()],
                 [y_test[:, i].min(), y_test[:, i].max()],
                 'r--', label='Ideal')
        plt.gca().text(0.05, 0.95, textstr, transform=plt.gca().transAxes,
                        verticalalignment='top',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
        plt.xlabel(f"Actual {name}")
        plt.ylabel(f"Predicted {name}")
        plt.title(f"{model}: Actual vs Predicted {name}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()

def plot_costs(costs_actual, costs_pred):
    plt.figure(figsize=(5, 4))
    plt.scatter(costs_actual, costs_pred, alpha=0.6)
    min_val = min(min(costs_actual), min(costs_pred))
    max_val = max(max(costs_actual), max(costs_pred))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Ideal')
    plt.xlabel("Actual costs")
    plt.ylabel("Predicted costs")
    plt.title("Actual vs Predicted costs")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_predictions_validity(y_test, y_pred, X_test, target_names=['b', 'd', 'Ast'], model="LR"):
    for i, name in enumerate(target_names):
        plt.figure(figsize=(5, 4))

        valid_x = []
        valid_y = []
        invalid_x = []
        invalid_y = []

        all_x = []
        all_y = []

        for j in range(len(y_pred)):
            b, d, Ast = y_pred[j]
            fck, fy, Mu = X_test[j]

            x_val = y_test[j, i]
            y_val = y_pred[j, i]

            all_x.append(x_val)
            all_y.append(y_val)

            if is_valid_design(b, d, Ast, fck, fy, Mu):
                valid_x.append(x_val)
                valid_y.append(y_val)
            else:
                invalid_x.append(x_val)
                invalid_y.append(y_val)

        # Scatter plots
        plt.scatter(valid_x, valid_y, alpha=0.6, label='Valid Design', color='blue')
        plt.scatter(invalid_x, invalid_y, alpha=0.6, label='Failed Design', color='red')

        # Ideal line
        min_val = min(valid_x + invalid_x + valid_y + invalid_y)
        max_val = max(valid_x + invalid_x + valid_y + invalid_y)
        plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='Ideal')

        # Accuracy metrics
        r2 = r2_score(all_x, all_y)
        rmse = root_mean_squared_error(all_x, all_y)
        textstr = f"R² = {r2:.4f}\nRMSE = {rmse:.2f}"

        plt.gca().text(0.05, 0.95, textstr, transform=plt.gca().transAxes,
                       verticalalignment='top',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

        plt.xlabel(f"Actual {name}")
        plt.ylabel(f"Predicted {name}")
        plt.title(f"{model}: Actual vs Predicted {name}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.show()


In [ ]:
def plot_costs_validity(y_true, y_pred, X_test, model="LR"):
    plt.figure(figsize=(6, 5))

    valid_x = []
    valid_y = []
    invalid_x = []
    invalid_y = []

    all_actual = []
    all_pred = []

    for i in range(len(y_pred)):
        b, d, Ast = y_pred[i]
        fck, fy, Mu = X_test[i]

        cost_actual = get_cost(*y_true[i])
        cost_pred = get_cost(b, d, Ast)

        all_actual.append(cost_actual)
        all_pred.append(cost_pred)

        if is_valid_design(b, d, Ast, fck, fy, Mu):
            valid_x.append(cost_actual)
            valid_y.append(cost_pred)
        else:
            invalid_x.append(cost_actual)
            invalid_y.append(cost_pred)

    # Scatter plots
    plt.scatter(valid_x, valid_y, alpha=0.6, label='Valid Design', color='blue')
    plt.scatter(invalid_x, invalid_y, alpha=0.6, label='Failed Design', color='red')

    # Ideal line
    min_val = min(all_actual + all_pred)
    max_val = max(all_actual + all_pred)
    plt.plot([min_val, max_val], [min_val, max_val], 'k--', label='Ideal')

    # Accuracy metrics
    cost_acc = get_costs_accuracy(all_actual, all_pred)
    # Annotate R² and RMSE
    textstr = f"Accuracy = {cost_acc:.4f}"
    plt.gca().text(0.05, 0.95, textstr, transform=plt.gca().transAxes,
                   verticalalignment='top',
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

    # Labels
    plt.xlabel("Actual costs")
    plt.ylabel("Predicted costs")
    plt.title(f"Actual vs Predicted Costs: {model}")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
def correct_invalid_predictions(y_pred, X_test,
                                max_attempts=50,
                                step_percent=0.01,
                                max_step_percent=0.2):
    corrected_preds = []

    for i in range(len(y_pred)):
        b, d, Ast = y_pred[i]
        fck, fy, Mu = X_test[i]

        # If already valid and b >= 200, no need to change
        if is_valid_design(b, d, Ast, fck, fy, Mu) and b >= 200:
            corrected_preds.append([b, d, Ast])
            continue

        found_valid = False
        min_cost = float('inf')
        best_combo = [b, d, Ast]  # fallback

        # Try increasing step percentages
        for p in np.arange(step_percent, max_step_percent + step_percent, step_percent):
            delta_b = b * p
            delta_d = d * p
            delta_Ast = Ast * p

            # Try combinations: perturb up/down for each variable
            for db in [-delta_b, 0, delta_b]:
                new_b = b + db
                if new_b < 200:
                    continue  # skip if b is less than 200

                for dd in [-delta_d, 0, delta_d]:
                    new_d = d + dd
                    for dAst in [-delta_Ast, 0, delta_Ast]:
                        new_Ast = Ast + dAst

                        if is_valid_design(new_b, new_d, new_Ast, fck, fy, Mu):
                            cost = get_cost(new_b, new_d, new_Ast)
                            if cost < min_cost:
                                min_cost = cost
                                best_combo = [new_b, new_d, new_Ast]
                                found_valid = True

            if found_valid:
                break  # Exit early once a valid low-cost solution is found

        # Enforce minimum b even if fallback is used
        best_combo[0] = max(best_combo[0], 200)
        corrected_preds.append(best_combo)

    return np.array(corrected_preds)


In [ ]:
def show_random_designs(y_test, y_pred, get_cost, n=5, random_seed=None):
    """
    Show a random sample of predictions vs actual designs in a DataFrame.

    Parameters:
        y_pred (ndarray): Predicted [b, d, Ast] values
        y_test (ndarray): Actual [b, d, Ast] values
        get_cost (function): Function to compute cost given b, d, Ast
        n (int): Number of random samples to show
        random_seed (int): Optional seed for reproducibility

    Returns:
        pd.DataFrame: DataFrame containing actual, predicted and cost info
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    indices = np.random.choice(len(y_pred), size=n, replace=False)

    records = []
    for idx in indices:
        b_true, d_true, Ast_true = y_test[idx]
        b_pred, d_pred, Ast_pred = y_pred[idx]

        cost_actual = get_cost(b_true, d_true, Ast_true)
        cost_pred = get_cost(b_pred, d_pred, Ast_pred)

        cost_accuracy = 1 - abs(cost_pred - cost_actual) / cost_actual

        records.append({
            'b_actual': round(b_true, 2),
            'd_actual': round(d_true, 2),
            'Ast_actual': round(Ast_true, 2),
            'b_pred': round(b_pred, 2),
            'd_pred': round(d_pred, 2),
            'Ast_pred': round(Ast_pred, 2),
            'Cost_actual': round(cost_actual, 2),
            'Cost_pred': round(cost_pred, 2),
            'Cost_accuracy': round(cost_accuracy * 100, 2)  # in %
        })

    return pd.DataFrame(records)
